# LizyML Widget Tutorial — Binary Classification

This tutorial demonstrates using LizyML Widget for a **binary classification** task
with the [Breast Cancer Wisconsin](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset) dataset.

You will learn:
1. Loading real-world data into the widget
2. Reviewing auto-detected settings
3. Configuring evaluation metrics (including `precision_at_k` with custom k)
4. Enabling calibration for probability estimates
5. Reviewing binary-specific metrics and plots (ROC curve, calibration, feature importance)
6. Hyperparameter tuning
7. Running inference with probability predictions

## 0. Installation

```bash
pip install lizyml-widget lizyml[plots,tuning,calibration,explain]
```

## 1. Load the Breast Cancer Dataset

This dataset contains 569 samples with 30 features computed from
digitized images of fine needle aspirates of breast masses.
The target is binary: malignant (0) or benign (1).

In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df = data.frame
print(f"Shape: {df.shape}")
print(f"Target distribution:\n{df['target'].value_counts()}")
print(f"  0 = malignant, 1 = benign")
df.head()

Shape: (569, 31)
Target distribution:
target
1    357
0    212
Name: count, dtype: int64
  0 = malignant, 1 = benign


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


## 2. Launch the Widget

In [2]:
from lizyml_widget import LizyWidget

w = LizyWidget()
w.load(df, target="target")
w

## 3. Data Tab — Auto-Detection

The widget auto-detects:
- **Task**: `binary` (2 unique target values)
- **CV**: `stratified_kfold` (preserves class balance)
- **Columns**: All 30 features included

In [3]:
print(f"Task:     {w.task}")
print(f"CV:       {w.cv_method} ({w.cv_n_splits} folds)")
print(f"Shape:    {w.df_shape}")

Task:     binary
CV:       stratified_kfold (5 folds)
Shape:    [569, 31]


## 4. Config — Model & Evaluation Settings

For binary classification, we configure:
- Multiple evaluation metrics including `precision_at_k` with custom k
- Calibration enabled (Platt scaling) for well-calibrated probabilities

In [4]:
w.set_config({
    "model": {
        "name": "lgbm",
        "params": {
            "n_estimators": 300,
            "learning_rate": 0.05,
            "max_depth": 4,
        },
    },
    "training": {
        "seed": 42,
        "early_stopping": {"enabled": True, "rounds": 50},
    },
    "evaluation": {
        "metrics": ["auc", "logloss", "f1", "precision_at_k"],
        "params": {"precision_at_k_k": 20},
    },
    "calibration": {
        "method": "platt",
        "n_splits": 5,
        "params": {},
    },
})

print("Config set with AUC, LogLoss, F1, precision_at_k (k=20), and Platt calibration")

Config set with AUC, LogLoss, F1, precision_at_k (k=20), and Platt calibration


## 5. Run Fit

In [5]:
w.fit()

## 6. Results — Binary Classification Metrics & Plots

After fitting, the Results tab shows:

**Metrics** (IS = in-sample, OOS = out-of-sample):
- **AUC** — Area Under ROC Curve
- **LogLoss** — Logarithmic Loss
- **F1** — F1 Score
- **precision_at_k (k=20)** — Precision among top-k predictions

**Binary-specific plots**:
- **ROC Curve** — Receiver Operating Characteristic
- **Calibration** — Predicted vs actual probability (when calibration enabled)
- **Probability Histogram** — Distribution of predicted probabilities
- **Feature Importance** — Split, Gain, and SHAP variants

In [6]:
summary = w.get_fit_summary()
if summary:
    print(f"Fold count: {summary.fold_count}")
    for name, vals in summary.metrics.items():
        if isinstance(vals, dict):
            oos = vals.get("oos", "N/A")
            print(f"  {name}: OOS={oos}")

Fold count: 5
  auc: OOS=0.971929337772845
  logloss: OOS=0.31802244258453705
  f1: OOS=0.8726355611601513
  precision_at_k: OOS=1.0


## 7. Tune — Hyperparameter Optimization

In [ ]:
w.tune()

tune_summary = w.get_tune_summary()
if tune_summary:
    print(f"Best score:  {tune_summary.best_score:.4f}")
    print(f"Metric:      {tune_summary.metric_name} ({tune_summary.direction})")
    print(f"Best params: {tune_summary.best_params}")
    print(f"Trials:      {len(tune_summary.trials)}")

## 8. Inference — Predict with Probabilities

For binary classification, predictions include probability estimates.

In [ ]:
df_test = df.drop(columns=["target"]).tail(10)

result = w.predict(df_test)
if result:
    print(f"Predictions shape: {result.predictions.shape}")
    print(f"Columns: {list(result.predictions.columns)}")
    result.predictions.head()

## 9. Save Config

In [ ]:
w.save_config("binary_config.yaml")
print("Config saved to binary_config.yaml")

## 10. Export Code

In [ ]:
# Export standalone training code (no LizyML dependency needed)
export_path = w.export_code("./exported_code")
print(f"Code exported to: {export_path}")
print("Generated files:")
for f in sorted(export_path.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(export_path)}")